# Phase 1: Data Exploration and analysis

**A note on authentication.** As of October 2026, F1 requires an F1TV account to access its public API so in order to use FastF1 you would need to create and sign in to your account. (No paid subscription required)

```
source .venv/bin/activate
python -m fastf1 auth f1tv --authenticate
```

This will prompt for your F1TV email and password directly in the terminal and store a login token locally. The token lasts about four days so this needs repeating periodically.

Ignore any error or warning logs when you run the notebook

In [ ]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt

# Local cache so a session never has to be downloaded twice.
fastf1.Cache.enable_cache("../data/fastf1_cache")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## How many races are in this season 2026

FastF1's event schedule gives one row per race weekend, including pre-season testing as round 0.

In [ ]:
schedule = fastf1.get_event_schedule(2026)
print("Number of races in 2026:", (schedule["RoundNumber"] > 0).sum())
schedule[["RoundNumber", "EventName", "Location", "EventDate"]]

## How many races FastF1 has in total, across every season

In [ ]:
season_race_counts = {}
for year in range(2018, 2027):
    sched = fastf1.get_event_schedule(year)
    season_race_counts[year] = int((sched["RoundNumber"] > 0).sum())

counts_df = pd.DataFrame(season_race_counts.items(), columns=["Season", "Races"]).set_index("Season")
print(counts_df)
print("Total races across all seasons FastF1 covers:", counts_df["Races"].sum())

## How many sprint races F1 has hosted

F1 introduced the sprint race (a shortened race about 30 laps or less) from 2021 onwards.
The current format of a sprint weekend is 
Friday: 1 Free practice session, Sprint Qualifying,
Saturday: Sprint Race, Qualifying,
Sunday: Race

In [ ]:
sprint_counts = {}
for year in range(2018, 2027):
    sched = fastf1.get_event_schedule(year)
    sprint_counts[year] = int(sched["EventFormat"].str.contains("sprint", na=False).sum())

sprint_df = pd.DataFrame(sprint_counts.items(), columns=["Season", "Sprints"]).set_index("Season")
print(sprint_df)
print("Total sprint weekends F1 has hosted:", sprint_df["Sprints"].sum())

## Who is the GOAT in the FastF1 era

Lets see who has the most race wins from 2018 to now

In [ ]:
from collections import Counter

win_counts = Counter()
today = pd.Timestamp.now()

for year in range(2018, 2027):
    sched = fastf1.get_event_schedule(year)
    rounds = sched[(sched["RoundNumber"] > 0) & (sched["EventDate"] < today)]["RoundNumber"]
    for rnd in rounds:
        s = fastf1.get_session(year, int(rnd), "R")
        s.load(laps=False, telemetry=False, weather=False, messages=False)
        winner = s.results[s.results["Position"] == 1].iloc[0]
        win_counts[winner["FullName"]] += 1

wins_df = pd.DataFrame(win_counts.most_common(10), columns=["Driver", "Wins"])
print(wins_df)

Max dominated in the Ground Effect era with that Red Bull. Simply Lovely :)

## Pull the 2026 Italian Grand Prix data

Monza is the fastest circuit on the calendar and one of the most iconic circuits (one that i went to in the 2025 Italian Grand Prix).
Lets do proper analysis on the data and look at what type of lap data there are.

In [ ]:
session_mz = fastf1.get_session(2026, "Monza", "R")
session_mz.load(telemetry=False, weather=False, messages=False)

## Look at the raw laps table

`session.laps` is one row per lap, per driver, for the whole race. This is the table every later cleaning and feature engineering step starts from, so it is worth seeing its actual shape and columns before anything gets filtered out of it.

In [ ]:
mz_laps = session_mz.laps
mz_total_laps = int(mz_laps["LapNumber"].max())
print("Shape:", mz_laps.shape)
mz_laps.head()

It is sorted by driver number so it shows Lando Norris data at the head since he was the 2025 F1 Champion and has the number "1"

Lets see what columns the FastF1 data has in the race and what data types they have

In [ ]:
mz_laps.dtypes

A lot of irrelevant columns lets look at the more important ones


In [ ]:
cols = ["Driver", "LapNumber", "LapTime", "Compound", "TyreLife", "Stint", "PitInTime", "PitOutTime", "TrackStatus"]
mz_laps[cols].head(15)

What do the TrackStatus codes mean

`TrackStatus` is a string of digits, one for each status that was active at some point during that lap, concatenated in the order they happened.

- `1` = Track clear
- `2` = Yellow flag
- `3` = unused ???
- `4` = Safety Car
- `5` = Red Flag
- `6` = Virtual Safety Car deployed
- `7` = Virtual Safety Car ending

So `1245` is a lap that went clear -> yellow -> Safety Car -> Red Flag, and `12` is just clear -> yellow, a normal yellow flag.
The below code shows all the unique track statuses that happened in the race.

In [ ]:
TRACK_STATUS_MEANINGS = {
    "1": "Track clear",
    "2": "Yellow flag",
    "4": "Safety Car",
    "5": "Red Flag",
    "6": "VSC deployed",
    "7": "VSC ending",
}

for status in sorted(mz_laps["TrackStatus"].unique()):
    decoded = " -> ".join(TRACK_STATUS_MEANINGS.get(c, c) for c in status)
    print(f"{status}: {decoded}")

## Practice: tyre degradation on long runs

A proper look at tyre degradation needs a genuine continuous run excluding in/out laps and cooldown laps. For each practice session: find stints of 8 laps or more, drop the in lap and out lap, then take the longest unbroken sequence of consecutive laps that are all within 107% of the session's fastest lap. 

In [ ]:
for fp in ["FP1", "FP2", "FP3"]:
    session_fp = fastf1.get_session(2026, "Monza", fp)
    session_fp.load(telemetry=False, weather=False, messages=False)
    laps = session_fp.laps.sort_values(["Driver", "Stint", "LapNumber"])

    threshold = laps["LapTime"].min() * 1.07
    stint_size = laps.groupby(["Driver", "Stint"])["LapNumber"].transform("size")
    stint_rank = laps.groupby(["Driver", "Stint"]).cumcount()

    middle = laps[(stint_size >= 6) & (stint_rank > 0) & (stint_rank < stint_size - 1)].copy()
    middle["within_107"] = middle["LapTime"] < threshold

    print(f"\n{fp} longest continuous sub-107% run per stint (in/out laps excluded, 8+ lap runs only):")
    for (driver, stint), stint_laps in middle.groupby(["Driver", "Stint"]):
        run_id = (stint_laps["within_107"] != stint_laps["within_107"].shift()).cumsum()
        clean_runs = [run for _, run in stint_laps[stint_laps["within_107"]].groupby(run_id)]
        if not clean_runs:
            continue
        best_run = max(clean_runs, key=len)
        if len(best_run) < 8:
            continue
        compound = best_run["Compound"].iloc[0]
        original_laps = int(stint_size.loc[stint_laps.index[0]])
        first_time = best_run["LapTime"].iloc[0].total_seconds()
        last_time = best_run["LapTime"].iloc[-1].total_seconds()
        deg_per_lap = (last_time - first_time) / (len(best_run) - 1)
        print(f"{driver} ({compound}, {original_laps}-lap stint, {len(best_run)}-lap continuous run): {first_time:.3f}s -> {last_time:.3f}s, deg {deg_per_lap:+.3f}s/lap")

Looking at the data the tyre deg is quite minimal for stints around 10 laps. Since Monza is a high speed circuit with not as many corners, this is expected and also when fuel loads burn off, the laps get faster. However we would see a significant tyre falloff in actual race stints where you would be on a tyre for 20 or more laps.

Lets look at the qualifying leaderboard

In [ ]:
session_q = fastf1.get_session(2026, "Monza", "Q")
session_q.load(laps=False, telemetry=False, weather=False, messages=False)
session_q.results[["Position", "Abbreviation", "Q1", "Q2", "Q3"]]